# Silver — ecommerce_itens_pedido

Este notebook lê a Bronze Delta `squad1.bronze_ecommerce_itens_pedido`, aplica as 10 regras de qualidade da tabela de itens de pedido, grava a Silver Delta e registra os resultados na tabela compartilhada `squad1.dq_monitoring_logs`.




## Imports e parâmetros

In [0]:
#Premissas:
#- A Bronze foi salva como tabela Delta gerenciada via `saveAsTable`.
#- A Silver mantém as linhas avaliadas e adiciona flags booleanas de falha por regra.
#- A tabela `dq_monitoring_logs` é criada automaticamente caso não exista.
#- O processamento evita duplicidade por `bronze_source_file`.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
import uuid

RUN_ID = str(uuid.uuid4())

TABELA_BRONZE_ITENS = "squad1.bronze_ecommerce_itens_pedido"
TABELA_BRONZE_PEDIDOS = "squad1.bronze_ecommerce_pedidos"
TABELA_BRONZE_PRODUTOS = "squad1.bronze_ecommerce_produtos"

TABELA_SILVER_ITENS = "squad1.silver_ecommerce_itens_pedido"
DQ_LOGS_TABLE = "squad1.dq_monitoring_logs"

print("RUN_ID:", RUN_ID)


## Funções auxiliares

In [0]:
def tabela_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def escolher_tabela_existente(*tabelas):
    for tabela in tabelas:
        if tabela_existe(tabela):
            print(f"Tabela encontrada: {tabela}")
            return tabela
    raise Exception(f"Nenhuma das tabelas existe: {tabelas}")


def garantir_dq_monitoring_logs():
    schema_dq_logs = StructType([
        StructField("run_id", StringType(), False),
        StructField("tabela", StringType(), False),
        StructField("regra", StringType(), False),
        StructField("status", StringType(), False),
        StructField("severidade", StringType(), False),
        StructField("qtd_registros_falhos", IntegerType(), False),
        StructField("qtd_registros_total", IntegerType(), False),
        StructField("timestamp_execucao", TimestampType(), False),
        StructField("arquivo_origem", StringType(), False),
    ])

    if not tabela_existe(DQ_LOGS_TABLE):
        df_empty = spark.createDataFrame([], schema_dq_logs)
        (
            df_empty.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(DQ_LOGS_TABLE)
        )
        print(f"Tabela criada: {DQ_LOGS_TABLE}")
    else:
        print(f"Tabela já existe: {DQ_LOGS_TABLE}")


def remover_arquivos_ja_processados(df_bronze, tabela_silver):
    if not tabela_existe(tabela_silver):
        print(f"Tabela Silver ainda não existe: {tabela_silver}. Todos os arquivos serão processados.")
        return df_bronze

    df_processados = (
        spark.table(tabela_silver)
        .select("bronze_source_file")
        .where(F.col("bronze_source_file").isNotNull())
        .dropDuplicates()
    )

    return df_bronze.join(df_processados, on="bronze_source_file", how="left_anti")


def criar_log_regra(df, nome_tabela, nome_regra, coluna_flag, severidade):
    return (
        df
        .groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag) == True, F.lit(1)).otherwise(F.lit(0))).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit(nome_tabela))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS")))
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id", "tabela", "regra", "status", "severidade",
            "qtd_registros_falhos", "qtd_registros_total",
            "timestamp_execucao", "arquivo_origem"
        )
    )


def gravar_logs_sem_duplicar(df_logs):
    garantir_dq_monitoring_logs()

    df_existentes = (
        spark.table(DQ_LOGS_TABLE)
        .select("tabela", "regra", "arquivo_origem")
        .dropDuplicates()
    )

    df_logs_novos = (
        df_logs
        .join(df_existentes, on=["tabela", "regra", "arquivo_origem"], how="left_anti")
    )

    qtd = df_logs_novos.count()

    if qtd == 0:
        print("Nenhum log novo para gravar em dq_monitoring_logs.")
        return

    (
        df_logs_novos.write
        .format("delta")
        .mode("append")
        .saveAsTable(DQ_LOGS_TABLE)
    )

    print(f"Logs gravados em {DQ_LOGS_TABLE}: {qtd}")


## Garantir existência da tabela de logs

In [0]:
garantir_dq_monitoring_logs()

## Ler Bronze e referências

In [0]:
if not tabela_existe(TABELA_BRONZE_ITENS):
    raise Exception(f"Tabela Bronze não encontrada: {TABELA_BRONZE_ITENS}")

df_itens_bronze = spark.table(TABELA_BRONZE_ITENS)

tabela_pedidos_ref = escolher_tabela_existente(
    "squad1.silver_ecommerce_pedidos",
    TABELA_BRONZE_PEDIDOS
)

tabela_produtos_ref = escolher_tabela_existente(
    "squad1.silver_ecommerce_produtos",
    TABELA_BRONZE_PRODUTOS
)

df_pedidos_ref = spark.table(tabela_pedidos_ref)
df_produtos_ref = spark.table(tabela_produtos_ref)

display(df_itens_bronze.limit(5))


## Selecionar apenas micro-lotes ainda não processados

In [0]:
df_itens_novos = remover_arquivos_ja_processados(
    df_bronze=df_itens_bronze,
    tabela_silver=TABELA_SILVER_ITENS
)

qtd_novos = df_itens_novos.count()
print("Registros novos para processar:", qtd_novos)

if qtd_novos == 0:
    dbutils.notebook.exit("Nenhum arquivo novo encontrado para processamento da Silver de itens_pedido.")


## Padronização mínima para validação

In [0]:
df_base = (
    df_itens_novos
    .withColumn("id_item_pedido", F.col("id_item_pedido").cast("long"))
    .withColumn("id_pedido", F.col("id_pedido").cast("long"))
    .withColumn("sku", F.trim(F.col("sku")))
    .withColumn("quantidade", F.col("quantidade").cast("double"))
    .withColumn("preco_unitario", F.col("preco_unitario").cast("double"))
    .withColumn("desconto_aplicado", F.col("desconto_aplicado").cast("double"))
)

colunas_pedidos = df_pedidos_ref.columns

select_pedidos = [
    F.col("id_pedido").cast("long").alias("id_pedido"),
    F.col("valor_total").cast("double").alias("valor_total")
]

if "bronze_source_file" in colunas_pedidos:
    select_pedidos.append(F.col("bronze_source_file").alias("pedido_bronze_source_file"))

df_pedidos_ref_base = (
    df_pedidos_ref
    .select(*select_pedidos)
    .dropDuplicates(["id_pedido"])
)

df_produtos_ref_base = (
    df_produtos_ref
    .select(F.trim(F.col("sku")).alias("sku"))
    .dropDuplicates(["sku"])
    .withColumn("sku_existe", F.lit(True))
)


##  Regras 1, 2 e 3 — PK e FKs

In [0]:
w_id_item = Window.partitionBy("id_item_pedido")

df_regras = (
    df_base
    .withColumn("qtd_id_item_pedido", F.count("*").over(w_id_item))
    .withColumn(
        "r1_id_item_pedido_falhou",
        F.col("id_item_pedido").isNull() | (F.col("qtd_id_item_pedido") > 1)
    )
)

df_pedidos_existentes = (
    df_pedidos_ref_base
    .select("id_pedido")
    .dropDuplicates()
    .withColumn("pedido_existe", F.lit(True))
)

df_regras = (
    df_regras
    .join(df_pedidos_existentes, on="id_pedido", how="left")
    .withColumn(
        "r2_id_pedido_fk_falhou",
        F.col("id_pedido").isNull() | F.col("pedido_existe").isNull()
    )
    .drop("pedido_existe")
)

df_regras = (
    df_regras
    .join(df_produtos_ref_base, on="sku", how="left")
    .withColumn(
        "r3_sku_fk_falhou",
        F.col("sku").isNull() | (F.col("sku") == "") | F.col("sku_existe").isNull()
    )
    .drop("sku_existe")
)


##  Regras 4, 5, 6, 8 e 9 — Valores de item

In [0]:
df_regras = (
    df_regras
    .withColumn(
        "r4_quantidade_falhou",
        F.col("quantidade").isNull()
        | (F.col("quantidade") < 1)
        | (F.col("quantidade") != F.floor(F.col("quantidade")))
    )
    .withColumn(
        "r5_preco_unitario_falhou",
        F.col("preco_unitario").isNull()
        | (F.col("preco_unitario") <= 0)
    )
    .withColumn(
        "r6_desconto_maior_preco_falhou",
        F.col("desconto_aplicado").isNotNull()
        & F.col("preco_unitario").isNotNull()
        & (F.col("desconto_aplicado") > F.col("preco_unitario"))
    )
    .withColumn(
        "r8_desconto_negativo_falhou",
        F.col("desconto_aplicado").isNull()
        | (F.col("desconto_aplicado") < 0)
    )
    .withColumn(
        "r9_percentual_desconto_falhou",
        F.col("preco_unitario").isNotNull()
        & (F.col("preco_unitario") > 0)
        & F.col("desconto_aplicado").isNotNull()
        & (F.col("desconto_aplicado") > (F.col("preco_unitario") * 0.5))
    )
)


## Regra 7 — Consistência financeira entre itens e pedidos

In [0]:
df_total_itens_por_pedido = (
    df_regras
    .withColumn(
        "valor_item_calculado",
        (F.col("preco_unitario") - F.coalesce(F.col("desconto_aplicado"), F.lit(0.0))) * F.col("quantidade")
    )
    .groupBy("id_pedido")
    .agg(F.sum("valor_item_calculado").alias("valor_total_itens_calculado"))
)

df_com_total_pedido = (
    df_total_itens_por_pedido
    .join(df_pedidos_ref_base.select("id_pedido", "valor_total"), on="id_pedido", how="left")
    .withColumn(
        "r7_total_pedido_falhou",
        F.col("valor_total").isNull()
        | F.col("valor_total_itens_calculado").isNull()
        | (F.abs(F.col("valor_total_itens_calculado") - F.col("valor_total")) > F.lit(0.01))
    )
    .select("id_pedido", "valor_total_itens_calculado", "valor_total", "r7_total_pedido_falhou")
)

df_regras = (
    df_regras
    .join(df_com_total_pedido, on="id_pedido", how="left")
    .withColumn("r7_total_pedido_falhou", F.coalesce(F.col("r7_total_pedido_falhou"), F.lit(True)))
)


##  Regra 10 — Cada pedido deve ter no mínimo 1 item

In [0]:
df_regras = df_regras.withColumn("r10_pedido_sem_item_falhou", F.lit(False))

select_pedidos_sem_item = ["id_pedido"]
if "pedido_bronze_source_file" in df_pedidos_ref_base.columns:
    select_pedidos_sem_item.append("pedido_bronze_source_file")

df_pedidos_sem_item = (
    df_pedidos_ref_base
    .select(*select_pedidos_sem_item)
    .join(df_regras.select("id_pedido").dropDuplicates(), on="id_pedido", how="left_anti")
)

qtd_pedidos_total = df_pedidos_ref_base.select("id_pedido").dropDuplicates().count()
qtd_pedidos_sem_item = df_pedidos_sem_item.count()

print("Pedidos totais na referência:", qtd_pedidos_total)
print("Pedidos sem item:", qtd_pedidos_sem_item)


## Criar Silver de itens

In [0]:
colunas_flags = [
    "r1_id_item_pedido_falhou",
    "r2_id_pedido_fk_falhou",
    "r3_sku_fk_falhou",
    "r4_quantidade_falhou",
    "r5_preco_unitario_falhou",
    "r6_desconto_maior_preco_falhou",
    "r7_total_pedido_falhou",
    "r8_desconto_negativo_falhou",
    "r9_percentual_desconto_falhou",
    "r10_pedido_sem_item_falhou"
]

condicao_alguma_falha = None

for coluna in colunas_flags:
    condicao = F.col(coluna) == True
    condicao_alguma_falha = condicao if condicao_alguma_falha is None else (condicao_alguma_falha | condicao)

df_silver_itens = (
    df_regras
    .withColumn("silver_linha_valida", F.when(condicao_alguma_falha, F.lit(False)).otherwise(F.lit(True)))
    .withColumn("silver_processed_at", F.current_timestamp())
    .withColumn("silver_run_id", F.lit(RUN_ID))
    .drop("qtd_id_item_pedido")
)

display(df_silver_itens.limit(10))


##  Gravar Silver Delta sem duplicar arquivos

In [0]:
(
    df_silver_itens.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABELA_SILVER_ITENS)
)

print(f"Silver gravada: {TABELA_SILVER_ITENS}")


##  Criar logs de DQ

In [0]:
logs = []

logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R1 - id_item_pedido não pode ser nulo nem duplicado", "r1_id_item_pedido_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R2 - id_pedido deve existir em ecommerce_pedidos", "r2_id_pedido_fk_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R3 - sku deve existir em ecommerce_produtos", "r3_sku_fk_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R4 - quantidade deve ser >= 1 e inteira", "r4_quantidade_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R5 - preco_unitario deve ser > 0", "r5_preco_unitario_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R6 - desconto_aplicado não pode ser maior que preco_unitario", "r6_desconto_maior_preco_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R7 - total dos itens deve bater com valor_total do pedido", "r7_total_pedido_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R8 - desconto_aplicado não pode ser negativo", "r8_desconto_negativo_falhou", "Critica"))
logs.append(criar_log_regra(df_silver_itens, "silver_ecommerce_itens_pedido", "R9 - percentual de desconto não deve exceder 50%", "r9_percentual_desconto_falhou", "Aviso"))

df_dq_monitoring_logs = logs[0]
for df_log in logs[1:]:
    df_dq_monitoring_logs = df_dq_monitoring_logs.unionByName(df_log)


## Log especial da Regra 10

In [0]:
if "pedido_bronze_source_file" in df_pedidos_ref_base.columns:
    df_logs_r10_base = (
        df_pedidos_ref_base
        .select(
            F.coalesce(F.col("pedido_bronze_source_file"), F.lit("ecommerce_pedidos")).alias("arquivo_origem"),
            "id_pedido"
        )
        .join(
            df_regras.select("id_pedido").dropDuplicates().withColumn("tem_item", F.lit(True)),
            on="id_pedido",
            how="left"
        )
        .groupBy("arquivo_origem")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col("tem_item").isNull(), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
    )
else:
    df_logs_r10_base = spark.createDataFrame(
        [("ecommerce_pedidos", int(qtd_pedidos_total), int(qtd_pedidos_sem_item))],
        ["arquivo_origem", "qtd_registros_total", "qtd_registros_falhos"]
    )

df_logs_r10 = (
    df_logs_r10_base
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("tabela", F.lit("silver_ecommerce_itens_pedido"))
    .withColumn("regra", F.lit("R10 - cada pedido deve ter no mínimo 1 item associado"))
    .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS")))
    .withColumn("severidade", F.lit("Critica"))
    .withColumn("timestamp_execucao", F.current_timestamp())
    .select(
        "run_id", "tabela", "regra", "status", "severidade",
        "qtd_registros_falhos", "qtd_registros_total",
        "timestamp_execucao", "arquivo_origem"
    )
)

df_dq_monitoring_logs = df_dq_monitoring_logs.unionByName(df_logs_r10)

display(df_dq_monitoring_logs)


## Gravar dq_monitoring_logs sem duplicidade

In [0]:
gravar_logs_sem_duplicar(df_dq_monitoring_logs)

## Validação final

In [0]:
print("=" * 80)
print("SILVER ecommerce_itens_pedido CONCLUÍDA")
print("Tabela Silver:", TABELA_SILVER_ITENS)
print("Tabela DQ Logs:", DQ_LOGS_TABLE)
print("Run ID:", RUN_ID)
print("Registros processados:", df_silver_itens.count())
print("=" * 80)

display(df_silver_itens.groupBy("silver_linha_valida").count())

display(
    spark.table(DQ_LOGS_TABLE)
    .filter(F.col("run_id") == RUN_ID)
    .orderBy("regra", "arquivo_origem")
)
